# Tensor Manipulation, Einops & Einsum

**What you'll learn:**
- Raw tensor operations (PyTorch): creation, shape, indexing, math
- Einops: `rearrange`, `reduce`, `repeat`  - declarative tensor restructuring
- Einsum: Einstein summation - tensor algebra and contractions
- When and why to use each tool
- Real ML use cases: Attention, ViT, CNN pooling, GNNs

**Mental Model:**
> Think of tensors as an organized warehouse with labeled shelves.
> - **Raw manipulation** → Move shelves around (no labels)
> - **Einops** → Reorganize shelves with clear labels
> - **Einsum** → Do math *across* shelves


### **PyTorch Tensor**

The ``Tensor`` class is very similar to numpy's ``ndarray`` and provides most of its functionality.


However, it also has two important distinctions:

- ``Tensor`` supports GPU computations.
- ``Tensor`` may store extra information needed for back-propagation:
  - The gradient tensor w.r.t. some variable (e.g. the loss)
  - A node representing an operation in the computational graph that produced this tensor.

- Note :  **tensor operations are not in-place**.

**Detailed Document Here**: [PyTorch docs](https://pytorch.org/docs/stable/).

## Setup & Imports

In [145]:
import torch
torch.__version__

'2.10.0+cpu'

In [146]:
# Install einops if not already installed
# !pip install einops

import torch
import numpy as np
from einops import rearrange, reduce, repeat
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# For pretty print tensors with shape and dtype
def show(tensor, name="tensor"):
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")
    print(f"{tensor}\n")

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.10.0+cpu


## PART 1: Raw Tensor Manipulation

It is the building blocks of all **tensor** operations.

## 1.1 Tensor Creation

### From data

In [147]:
t_from_list = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
show(t_from_list, "from_list")

from_list: shape=(2, 3), dtype=torch.float32
tensor([[1., 2., 3.],
        [4., 5., 6.]])



### Filled tensors

In [148]:
zeros   = torch.zeros(3, 4)
ones    = torch.ones(2, 3)
full    = torch.full((2, 2), fill_value=7.0)
eye     = torch.eye(4)           # identity matrix
show(zeros, "zeros(3,4)")
show(eye,   "eye(4)")

zeros(3,4): shape=(3, 4), dtype=torch.float32
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

eye(4): shape=(4, 4), dtype=torch.float32
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])



### Sequences

In [149]:
arange  = torch.arange(0, 10, 2)         # [0,2,4,6,8]
linspace= torch.linspace(0, 1, steps=5)  # [0, 0.25, 0.5, 0.75, 1]
show(arange,   "arange(0,10,2)")
show(linspace, "linspace(0,1,5)")

arange(0,10,2): shape=(5,), dtype=torch.int64
tensor([0, 2, 4, 6, 8])

linspace(0,1,5): shape=(5,), dtype=torch.float32
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])



### Random tensors

In [150]:
rand    = torch.rand(3, 3)               # uniform [0,1)
randn   = torch.randn(3, 3)             # standard normal N(0,1)
randint = torch.randint(0, 10, (3, 3))  # random integers
show(randn,   "randn(3,3)")

randn(3,3): shape=(3, 3), dtype=torch.float32
tensor([[-1.3286, -1.7639,  0.4110],
        [-1.1337, -0.8581,  0.4271],
        [-0.9980,  0.3569, -0.0095]])



### Like (same shape, new values)

In [151]:
x = torch.randn(2, 5)
show(torch.zeros_like(x), "zeros_like(x)")
show(torch.ones_like(x),  "ones_like(x)")


zeros_like(x): shape=(2, 5), dtype=torch.float32
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

ones_like(x): shape=(2, 5), dtype=torch.float32
tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])



### From NumPy

In [152]:
np_arr  = np.array([[1.0, 2.0], [3.0, 4.0]])
from_np = torch.from_numpy(np_arr)
back_np = from_np.numpy()   # back to numpy (shares memory!)
show(from_np, "from_numpy")

from_numpy: shape=(2, 2), dtype=torch.float64
tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64)



## 1.2 Tensor Properties & Device Management

In [153]:
x = torch.randn(3, 4, 5)

print("=== Tensor Properties ===")
print(f"  shape   : {x.shape}          → (3, 4, 5)")
print(f"  ndim    : {x.ndim}           → number of dimensions")
print(f"  numel   : {x.numel()}        → total elements = 3×4×5=60")
print(f"  dtype   : {x.dtype}")
print(f"  device  : {x.device}")
print(f"  requires_grad: {x.requires_grad}")

=== Tensor Properties ===
  shape   : torch.Size([3, 4, 5])          → (3, 4, 5)
  ndim    : 3           → number of dimensions
  numel   : 60        → total elements = 3×4×5=60
  dtype   : torch.float32
  device  : cpu
  requires_grad: False


### Scalar extraction

In [154]:
scalar_tensor = torch.tensor(42.0)
print(f"\n  scalar.item() = {scalar_tensor.item()}  (converts to Python float)")


  scalar.item() = 42.0  (converts to Python float)


### Type casting

In [155]:
x_int   = x.to(torch.int32)
x_half  = x.to(torch.float16)   # half precision (useful for GPU training)
print(f"\n  x.to(int32): {x_int.dtype}")
print(f"  x.to(float16): {x_half.dtype}")


  x.to(int32): torch.int32
  x.to(float16): torch.float16


### Device management

In [156]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n  Active device: {device}")
x_device = x.to(device)
print(f"  x.to(device): {x_device.device}")


  Active device: cpu
  x.to(device): cpu


## 1.3 Shape Manipulation — reshape, view, permute, transpose

In [157]:
x = torch.arange(24).float()
show(x, "x (flat)")


x (flat): shape=(24,), dtype=torch.float32
tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23.])



### reshape
- Rows read left-to-right, unrolled then re-packed

In [158]:
x_2d  = x.reshape(4, 6)
x_3d  = x.reshape(2, 3, 4)
x_auto= x.reshape(6, -1)   # -1 = "figure it out" = 4
show(x_2d,  "reshape(4,6)")
show(x_3d,  "reshape(2,3,4)")

reshape(4,6): shape=(4, 6), dtype=torch.float32
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
        [ 6.,  7.,  8.,  9., 10., 11.],
        [12., 13., 14., 15., 16., 17.],
        [18., 19., 20., 21., 22., 23.]])

reshape(2,3,4): shape=(2, 3, 4), dtype=torch.float32
tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])



### view vs reshape
- view: guaranteed no-copy (shares memory), requires contiguous tensor
- reshape: copies if needed

In [159]:
x_view = x.view(3, 8)
print("view shares memory with x:", x_view.data_ptr() == x.data_ptr())

view shares memory with x: True


### flatten / unflatten

In [160]:
x_3d_sample = torch.randn(2, 3, 4)
flat   = x_3d_sample.flatten()              # all dims → 1D
flat_23= x_3d_sample.flatten(start_dim=1)   # keep batch, flatten rest
show(flat,    "flatten()")
show(flat_23, "flatten(start_dim=1)")

flatten(): shape=(24,), dtype=torch.float32
tensor([ 0.6938,  0.5269, -0.4401,  1.9864,  0.3864,  0.7721,  0.4957,  0.2821,
         0.9139,  1.6094,  0.5145, -0.5024,  0.0571, -0.1691, -0.2466, -0.3313,
        -0.3527, -1.4923, -0.9230,  0.6352,  0.3126,  0.3002, -0.1246, -0.0143])

flatten(start_dim=1): shape=(2, 12), dtype=torch.float32
tensor([[ 0.6938,  0.5269, -0.4401,  1.9864,  0.3864,  0.7721,  0.4957,  0.2821,
          0.9139,  1.6094,  0.5145, -0.5024],
        [ 0.0571, -0.1691, -0.2466, -0.3313, -0.3527, -1.4923, -0.9230,  0.6352,
          0.3126,  0.3002, -0.1246, -0.0143]])



### unsqueeze / squeeze

In [161]:
v = torch.randn(10)          # shape (10,)
v_col = v.unsqueeze(-1)      # (10, 1) — column vector
v_row = v.unsqueeze(0)       # (1, 10) — row vector
show(v_col, "unsqueeze(-1)")
show(v_row, "unsqueeze(0)")
show(v_col.squeeze(), "squeeze back")

unsqueeze(-1): shape=(10, 1), dtype=torch.float32
tensor([[-0.5767],
        [-0.4077],
        [ 0.7237],
        [ 0.7114],
        [-0.3851],
        [ 0.2437],
        [ 1.6107],
        [ 0.0588],
        [ 0.6586],
        [-0.1566]])

unsqueeze(0): shape=(1, 10), dtype=torch.float32
tensor([[-0.5767, -0.4077,  0.7237,  0.7114, -0.3851,  0.2437,  1.6107,  0.0588,
          0.6586, -0.1566]])

squeeze back: shape=(10,), dtype=torch.float32
tensor([-0.5767, -0.4077,  0.7237,  0.7114, -0.3851,  0.2437,  1.6107,  0.0588,
         0.6586, -0.1566])



### transpose / permute

In [162]:
mat = torch.randn(3, 5)
show(mat.T, "mat.T (2D transpose)")
show(mat.transpose(0,1), "transpose(0,1)")

mat.T (2D transpose): shape=(5, 3), dtype=torch.float32
tensor([[-1.1809, -0.0788,  1.2229],
        [ 0.0435, -1.6249, -1.0307],
        [ 0.4345, -0.5394,  0.3301],
        [ 0.3619,  0.4509, -1.1158],
        [ 0.7522, -1.4921, -0.4018]])

transpose(0,1): shape=(5, 3), dtype=torch.float32
tensor([[-1.1809, -0.0788,  1.2229],
        [ 0.0435, -1.6249, -1.0307],
        [ 0.4345, -0.5394,  0.3301],
        [ 0.3619,  0.4509, -1.1158],
        [ 0.7522, -1.4921, -0.4018]])



### permute: reorder ALL dims at once (powerful than transpose)

In [163]:
x4d = torch.randn(2, 3, 4, 5)   # (batch, channels, H, W)
x4d_permuted = x4d.permute(0, 2, 3, 1)  # → (batch, H, W, channels)
show(x4d_permuted, "permute(0,2,3,1): BCHW→BHWC")

permute(0,2,3,1): BCHW→BHWC: shape=(2, 4, 5, 3), dtype=torch.float32
tensor([[[[ 1.5350, -1.0599,  0.4548],
          [ 0.2050, -0.4841, -0.3869],
          [ 0.8037, -0.2216, -1.3440],
          [ 1.2058,  0.2628, -2.7597],
          [ 0.7017, -0.7040,  0.6078]],

         [[ 0.7781, -0.1914, -1.1261],
          [ 0.4588, -1.7681,  1.0384],
          [-0.4177,  1.0208, -0.7510],
          [ 0.3218,  1.5543,  1.1492],
          [-1.6302,  0.2835, -3.1647]],

         [[ 0.6581, -0.0504, -0.8033],
          [ 0.0776,  0.3901,  0.3081],
          [-1.5195, -0.1973,  0.5116],
          [-1.3644, -1.4866,  2.2310],
          [-1.3794,  1.3783,  0.3520]],

         [[-0.4311,  0.1662, -1.9663],
          [ 1.2203, -1.6969,  0.8346],
          [ 0.4007, -0.1712, -0.2023],
          [ 0.1999,  0.1036, -0.3045],
          [-2.3766,  2.3689,  0.7209]]],


        [[[-0.4362,  0.7520, -0.1235],
          [-1.6877, -1.6244,  0.2931],
          [-3.5045,  2.3918, -0.4782],
          [ 1.2675, -0.3

## 1.4 Combining Tensors — cat, stack, chunk, split

In [164]:
a = torch.arange(6).reshape(2, 3).float()
b = torch.arange(6, 12).reshape(2, 3).float()
show(a, "a"); show(b, "b")


a: shape=(2, 3), dtype=torch.float32
tensor([[0., 1., 2.],
        [3., 4., 5.]])

b: shape=(2, 3), dtype=torch.float32
tensor([[ 6.,  7.,  8.],
        [ 9., 10., 11.]])



### cat: joins along existing dim (no new dim added)

In [165]:
cat_dim0 = torch.cat([a, b], dim=0)   # (4, 3): stack rows
cat_dim1 = torch.cat([a, b], dim=1)   # (2, 6): stack columns
show(cat_dim0, "cat(dim=0) → (4,3)")
show(cat_dim1, "cat(dim=1) → (2,6)")

cat(dim=0) → (4,3): shape=(4, 3), dtype=torch.float32
tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.]])

cat(dim=1) → (2,6): shape=(2, 6), dtype=torch.float32
tensor([[ 0.,  1.,  2.,  6.,  7.,  8.],
        [ 3.,  4.,  5.,  9., 10., 11.]])



### stack: creates a NEW dimension

In [166]:
stack_dim0 = torch.stack([a, b], dim=0)  # (2, 2, 3)
stack_dim1 = torch.stack([a, b], dim=1)  # (2, 2, 3) — interleaved
show(stack_dim0, "stack(dim=0) → (2,2,3)")
show(stack_dim1, "stack(dim=1) → (2,2,3)")

stack(dim=0) → (2,2,3): shape=(2, 2, 3), dtype=torch.float32
tensor([[[ 0.,  1.,  2.],
         [ 3.,  4.,  5.]],

        [[ 6.,  7.,  8.],
         [ 9., 10., 11.]]])

stack(dim=1) → (2,2,3): shape=(2, 2, 3), dtype=torch.float32
tensor([[[ 0.,  1.,  2.],
         [ 6.,  7.,  8.]],

        [[ 3.,  4.,  5.],
         [ 9., 10., 11.]]])



### chunk: split into N equal pieces

In [167]:
x = torch.arange(12).reshape(3, 4).float()
chunks = torch.chunk(x, chunks=2, dim=1)  # split columns into 2
print("chunk shapes:", [c.shape for c in chunks])

chunk shapes: [torch.Size([3, 2]), torch.Size([3, 2])]


### split: split into specific sizes

In [168]:
parts = torch.split(x, split_size_or_sections=[1, 2], dim=0)
print("split shapes:", [p.shape for p in parts])

split shapes: [torch.Size([1, 4]), torch.Size([2, 4])]


## 1.5 Indexing & Slicing

In [169]:
x = torch.arange(20).reshape(4, 5).float()
show(x, "x (4×5)")

x (4×5): shape=(4, 5), dtype=torch.float32
tensor([[ 0.,  1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.,  9.],
        [10., 11., 12., 13., 14.],
        [15., 16., 17., 18., 19.]])



### Basic indexing

In [170]:
print("x[0]    =", x[0])           # first row
print("x[-1]   =", x[-1])          # last row
print("x[1, 3] =", x[1, 3])        # element at row=1, col=3
print("x[0, -1]=", x[0, -1])       # last element of first row

x[0]    = tensor([0., 1., 2., 3., 4.])
x[-1]   = tensor([15., 16., 17., 18., 19.])
x[1, 3] = tensor(8.)
x[0, -1]= tensor(4.)


### Slicing

In [171]:
print("\nx[:2, :]  =\n", x[:2, :])    # first 2 rows
print("x[:, 1:3] =\n", x[:, 1:3])   # columns 1 and 2
print("x[::2, ::2]=\n", x[::2, ::2]) # every other row AND col


x[:2, :]  =
 tensor([[0., 1., 2., 3., 4.],
        [5., 6., 7., 8., 9.]])
x[:, 1:3] =
 tensor([[ 1.,  2.],
        [ 6.,  7.],
        [11., 12.],
        [16., 17.]])
x[::2, ::2]=
 tensor([[ 0.,  2.,  4.],
        [10., 12., 14.]])


### Boolean (mask) indexing

In [172]:
mask = x > 10
print("\nElements > 10:", x[mask])   # 1D result of matching values
x_clipped = x.clone()
x_clipped[mask] = 0   # set all >10 to zero (in-place via mask)
print("After zeroing >10:\n", x_clipped)


Elements > 10: tensor([11., 12., 13., 14., 15., 16., 17., 18., 19.])
After zeroing >10:
 tensor([[ 0.,  1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.,  9.],
        [10.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.]])


### Integer array indexing

In [173]:
rows = torch.tensor([0, 2, 3])
cols = torch.tensor([1, 4, 0])
selected = x[rows, cols]   # gathers (0,1), (2,4), (3,0)
print("\nGather (row,col) pairs:", selected)


Gather (row,col) pairs: tensor([ 1., 14., 15.])


### Advanced: gather & scatter
- gather: select along one dim using index tensor

In [174]:
src   = torch.randn(3, 4)
idx   = torch.tensor([[0, 2, 1, 3], [3, 0, 2, 1], [1, 3, 0, 2]])
out   = torch.gather(src, dim=1, index=idx)
print("\ngather result shape:", out.shape)


gather result shape: torch.Size([3, 4])


## 1.6 Math Operations

In [175]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

### Element-wise

In [176]:
print("a + b =\n", a + b)
print("a * b =\n", a * b)     # NOT matrix mult, element-wise!
print("a ** 2 =\n", a ** 2)
print("torch.sqrt(a) =\n", torch.sqrt(a))
print("torch.exp(a) =\n", torch.exp(a))

a + b =
 tensor([[ 6.,  8.],
        [10., 12.]])
a * b =
 tensor([[ 5., 12.],
        [21., 32.]])
a ** 2 =
 tensor([[ 1.,  4.],
        [ 9., 16.]])
torch.sqrt(a) =
 tensor([[1.0000, 1.4142],
        [1.7321, 2.0000]])
torch.exp(a) =
 tensor([[ 2.7183,  7.3891],
        [20.0855, 54.5981]])


### Matrix multiplication

In [177]:
print("\na @ b (matmul) =\n", a @ b)             # matrix multiply
print("torch.mm(a,b)  =\n", torch.mm(a, b))      # same
print("torch.matmul   =\n", torch.matmul(a, b))  # most general


a @ b (matmul) =
 tensor([[19., 22.],
        [43., 50.]])
torch.mm(a,b)  =
 tensor([[19., 22.],
        [43., 50.]])
torch.matmul   =
 tensor([[19., 22.],
        [43., 50.]])


### Batch matrix multiply

In [178]:
A = torch.randn(8, 3, 4)  # batch of 8 matrices (3×4)
B = torch.randn(8, 4, 5)  # batch of 8 matrices (4×5)
C = torch.bmm(A, B)       # → (8, 3, 5)
print(f"\nbmm: ({A.shape}) × ({B.shape}) → {C.shape}")


bmm: (torch.Size([8, 3, 4])) × (torch.Size([8, 4, 5])) → torch.Size([8, 3, 5])


### Reductions

In [179]:
x = torch.arange(12.).reshape(3, 4)
show(x, "x")
print("sum all    :", x.sum().item())
print("sum dim=0  :", x.sum(dim=0))    # sum across rows → shape (4,)
print("sum dim=1  :", x.sum(dim=1))    # sum across cols → shape (3,)
print("mean all   :", x.mean().item())
print("max all    :", x.max().item())
print("argmax dim0:", x.argmax(dim=0)) # index of max per column
print("min dim=1  :", x.min(dim=1))    # (values, indices)

x: shape=(3, 4), dtype=torch.float32
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

sum all    : 66.0
sum dim=0  : tensor([12., 15., 18., 21.])
sum dim=1  : tensor([ 6., 22., 38.])
mean all   : 5.5
max all    : 11.0
argmax dim0: tensor([2, 2, 2, 2])
min dim=1  : torch.return_types.min(
values=tensor([0., 4., 8.]),
indices=tensor([0, 0, 0]))


### keepdim: preserve tensor rank after reduction

In [180]:
mean_kept = x.mean(dim=1, keepdim=True)  # (3,1) not (3,)
show(mean_kept, "mean(dim=1, keepdim=True)")

mean(dim=1, keepdim=True): shape=(3, 1), dtype=torch.float32
tensor([[1.5000],
        [5.5000],
        [9.5000]])



### In-place operations (denoted by _)
- In-place ops break autograd — avoid in model forward passes


In [181]:
y = torch.ones(3)
y.add_(2)    # in-place: y = y + 2
y.mul_(3)    # in-place: y = y * 3
print("\nAfter in-place add_(2) then mul_(3):", y)


After in-place add_(2) then mul_(3): tensor([9., 9., 9.])


## 1.7 Broadcasting
- Broadcasting: PyTorch automatically expands smaller tensors
- Rule: dimensions are aligned from the right; 1s are stretched.

### Scalar broadcast

In [182]:
x = torch.ones(3, 4)
y = x + 10   # 10 → broadcast to (3,4)
print(x)
print("x + 10:\n", y)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
x + 10:
 tensor([[11., 11., 11., 11.],
        [11., 11., 11., 11.],
        [11., 11., 11., 11.]])


### Vector broadcast

In [183]:
matrix = torch.zeros(3, 4)
bias   = torch.tensor([1., 2., 3., 4.])   # shape (4,)
out    = matrix + bias  # bias stretches to (3, 4)
show(out, "matrix + bias")

matrix + bias: shape=(3, 4), dtype=torch.float32
tensor([[1., 2., 3., 4.],
        [1., 2., 3., 4.],
        [1., 2., 3., 4.]])



### Column + Row broadcast

In [184]:
col = torch.tensor([[1.], [2.], [3.]])    # (3, 1)
row = torch.tensor([[10., 20., 30.]])     # (1, 3)
outer = col + row   # → (3, 3): all pairs summed
show(outer, "(3,1) + (1,3) → (3,3)")

(3,1) + (1,3) → (3,3): shape=(3, 3), dtype=torch.float32
tensor([[11., 21., 31.],
        [12., 22., 32.],
        [13., 23., 33.]])



### Real use: normalize each sample

In [185]:
batch  = torch.randn(32, 512)           # (batch, features)
mean   = batch.mean(dim=1, keepdim=True)  # (32, 1)
std    = batch.std(dim=1, keepdim=True)   # (32, 1)
normed = (batch - mean) / (std + 1e-8)    # broadcast over features
print(f"Normalized batch: mean≈{normed.mean():.4f}, std≈{normed.std():.4f}")

Normalized batch: mean≈0.0000, std≈0.9991


### Broadcasting failure example

In [186]:
try:
    a = torch.randn(3, 4)
    b = torch.randn(4, 3)
    c = a + b   # fails: shapes incompatible
except RuntimeError as e:
    print(f"\n Broadcasting error: {e}")


 Broadcasting error: The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1


## 1.8 Memory: Contiguous, Strides & Clone

### Strides tell PyTorch how many memory steps to take per dimension

In [187]:
x = torch.randn(3, 4)
print("Original strides:", x.stride())   # (4, 1): 4 steps for row, 1 for col
print("Is contiguous:", x.is_contiguous())

Original strides: (4, 1)
Is contiguous: True


### After transpose, strides flip but memory layout doesn't change

In [188]:
xt = x.T
print("\nTransposed strides:", xt.stride())  # (1, 4)
print("Is contiguous:", xt.is_contiguous())  # False!


Transposed strides: (1, 4)
Is contiguous: False


### .contiguous() forces a copy into standard row-major layout

In [189]:
xt_cont = xt.contiguous()
print("After .contiguous():", xt_cont.is_contiguous())  # True

After .contiguous(): True


### clone vs detach

In [190]:
a = torch.randn(3, 3, requires_grad=True)
b = a.clone()       # deep copy, keeps grad_fn
c = a.detach()      # shares memory, no gradients
d = a.detach().clone()  # safe copy with no gradients (most common pattern)
print("\nclone requires_grad  :", b.requires_grad)  # True
print("detach requires_grad :", c.requires_grad) # False


clone requires_grad  : True
detach requires_grad : False


---
# PART 2: Einops - Declarative Tensor Restructuring
---

Einops answers: **how is my tensor structured?**

Instead of tracking axis numbers, It give every dimension a **name**.

Three functions cover everything:
| Function | What it does |
|---|---|
| `rearrange` | Reshape, transpose, split, merge dims |
| `reduce` | Pool/aggregate: mean, sum, max, min |
| `repeat` | Expand/tile a tensor |


## 2.1 `rearrange` — Reshape + Transpose + Split + Merge

In [191]:
from einops import rearrange

x = torch.randn(2, 3, 4)  # (batch, height, width)
print(x)

tensor([[[-0.9110, -0.9122, -1.2428, -0.7777],
         [ 0.5119, -0.5177,  1.8133, -0.3615],
         [ 0.0897,  0.1561,  0.9899, -0.0598]],

        [[-1.0725,  0.0317, -0.3038, -0.4850],
         [ 0.2177,  0.3306,  0.3931,  0.2914],
         [-0.1522,  0.9447, -0.3339,  1.4655]]])


### Simple transpose

In [192]:
out = rearrange(x, 'b h w -> b w h')   # swap h and w
print(f"Transpose h,w:  {x.shape} → {out.shape}")

Transpose h,w:  torch.Size([2, 3, 4]) → torch.Size([2, 4, 3])


### Flatten dimensions

In [193]:
out = rearrange(x, 'b h w -> b (h w)')
print(f"Flatten h,w:    {x.shape} → {out.shape}")

out = rearrange(x, 'b h w -> (b h) w')
print(f"Merge batch+h:  {x.shape} → {out.shape}")

out = rearrange(x, 'b h w -> (b h w)')
print(f"Flatten all:    {x.shape} → {out.shape}")

Flatten h,w:    torch.Size([2, 3, 4]) → torch.Size([2, 12])
Merge batch+h:  torch.Size([2, 3, 4]) → torch.Size([6, 4])
Flatten all:    torch.Size([2, 3, 4]) → torch.Size([24])


### SPLIT a dimension (einops unique: raw .view() can't express this cleanly)

In [194]:
x2 = torch.randn(2, 12)   # (batch, hidden)
out = rearrange(x2, 'b (h d) -> b h d', h=3)   # split hidden into 3 heads of size 4
print(f"\nSplit hidden→heads: {x2.shape} → {out.shape}  [h=3, d=4]")


Split hidden→heads: torch.Size([2, 12]) → torch.Size([2, 3, 4])  [h=3, d=4]


### MERGE dimensions

In [195]:
x3 = torch.randn(2, 3, 4)  # (batch, heads, head_dim)
out = rearrange(x3, 'b h d -> b (h d)')   # merge heads back
print(f"Merge h,d: {x3.shape} → {out.shape}")

Merge h,d: torch.Size([2, 3, 4]) → torch.Size([2, 12])


### Add/remove singleton dims

In [196]:
v = torch.randn(10)
out = rearrange(v, 'd -> 1 d 1')   # add batch + trailing dim
print(f"\nAdd singleton: {v.shape} → {out.shape}")


Add singleton: torch.Size([10]) → torch.Size([1, 10, 1])


### Complex: BCHW → BHWC (image format conversion)

In [197]:
img = torch.randn(4, 3, 32, 32)   # (batch, channels, H, W) — PyTorch format
out = rearrange(img, 'b c h w -> b h w c')  # → TensorFlow / numpy format
print(f"\nBCHW→BHWC: {img.shape} → {out.shape}")


BCHW→BHWC: torch.Size([4, 3, 32, 32]) → torch.Size([4, 32, 32, 3])


# ViT Patchify: the most famous einops example
- Split an image into non-overlapping patches

In [198]:
images = torch.randn(8, 3, 224, 224)  # (batch, channels, H, W)
patch_size = 16

### RAW (cryptic)

In [199]:
B, C, H, W = images.shape
raw = images.reshape(B, C, H//patch_size, patch_size, W//patch_size, patch_size)
raw = raw.permute(0, 2, 4, 1, 3, 5)
raw = raw.reshape(B, -1, C * patch_size * patch_size)
print(f"RAW patchify result: {raw.shape}")

RAW patchify result: torch.Size([8, 196, 768])


### EINOPS (readable)

In [200]:
patches = rearrange(
    images,
    'b c (h p1) (w p2) -> b (h w) (c p1 p2)',
    p1=patch_size, p2=patch_size
)
print(f"Einops patchify result: {patches.shape}")

Einops patchify result: torch.Size([8, 196, 768])


### Both are identical — einops is just readable

In [201]:
print(f"Results match: {torch.allclose(raw, patches)}")
print(f"\n Input: (batch={B}, channels={C}, H=224, W=224)")
print(f" Output: (batch={B}, num_patches={14*14}, patch_pixels={C*patch_size*patch_size})")
print(f" = {B} images with {14*14} patches, each patch has {C*patch_size*patch_size} values")

Results match: True

 Input: (batch=8, channels=3, H=224, W=224)
 Output: (batch=8, num_patches=196, patch_pixels=768)
 = 8 images with 196 patches, each patch has 768 values


# Multi-Head Attention: split & merge heads

In [202]:
B, N, D = 32, 128, 512   # batch, seq_len, embed_dim
num_heads = 8
head_dim = D // num_heads  # 64

Q = torch.randn(B, N, D)  # (batch, seq, embed)

### Split embed_dim into heads
- (b n (h d)) means: the last dim is a product of h*d, split it

In [203]:
Q_heads = rearrange(Q, 'b n (h d) -> b h n d', h=num_heads)
print(f"Q: {Q.shape} → split heads → {Q_heads.shape}")
print(f"  Reading: batch={B} heads={num_heads} seq={N} head_dim={head_dim}")

Q: torch.Size([32, 128, 512]) → split heads → torch.Size([32, 8, 128, 64])
  Reading: batch=32 heads=8 seq=128 head_dim=64


### Merge heads back after attention

In [204]:
Q_merged = rearrange(Q_heads, 'b h n d -> b n (h d)')
print(f"\nQ_heads: {Q_heads.shape} → merge heads → {Q_merged.shape}")
print(f"Round-trip matches: {torch.allclose(Q, Q_merged)}")


Q_heads: torch.Size([32, 8, 128, 64]) → merge heads → torch.Size([32, 128, 512])
Round-trip matches: True


## 2.2 `reduce` — Named Aggregation

In [205]:
from einops import reduce

x = torch.randn(4, 8, 16, 16)  # (batch, channels, H, W) — CNN feature maps

### Basic reductions
- Global average pooling: collapse H and W

In [206]:
gap = reduce(x, 'b c h w -> b c', 'mean')
print(f"Global avg pool:   {x.shape} → {gap.shape}")

Global avg pool:   torch.Size([4, 8, 16, 16]) → torch.Size([4, 8])


### Global max pooling

In [207]:
gmp = reduce(x, 'b c h w -> b c', 'max')
print(f"Global max pool:   {x.shape} → {gmp.shape}")

Global max pool:   torch.Size([4, 8, 16, 16]) → torch.Size([4, 8])


### Spatial sum (keep channels)

In [208]:
spatial_sum = reduce(x, 'b c h w -> b c', 'sum')
print(f"Spatial sum:       {x.shape} → {spatial_sum.shape}")

Spatial sum:       torch.Size([4, 8, 16, 16]) → torch.Size([4, 8])


### Collapse everything: single scalar per batch

In [209]:
per_sample = reduce(x, 'b c h w -> b', 'mean')
print(f"Per-sample mean:   {x.shape} → {per_sample.shape}")

Per-sample mean:   torch.Size([4, 8, 16, 16]) → torch.Size([4])


### Global scalar

In [210]:
scalar = reduce(x, 'b c h w -> ', 'mean')
print(f"Global scalar:     {x.shape} → scalar {scalar.item():.4f}")

Global scalar:     torch.Size([4, 8, 16, 16]) → scalar 0.0337


### 2×2 Average Pooling (non-overlapping)
- This is what classic CNN max-pooling/avg-pooling does!

In [211]:
pool2 = reduce(x, 'b c (h p1) (w p2) -> b c h w', 'mean', p1=2, p2=2)
print(f"\n2×2 avg pooling:   {x.shape} → {pool2.shape}")


2×2 avg pooling:   torch.Size([4, 8, 16, 16]) → torch.Size([4, 8, 8, 8])


### Channel-wise reduction
- Average across channels (like grayscale conversion)

In [212]:
gray = reduce(x, 'b c h w -> b h w', 'mean')
print(f"Channel avg:       {x.shape} → {gray.shape}")

Channel avg:       torch.Size([4, 8, 16, 16]) → torch.Size([4, 16, 16])


### Compare with raw
- Raw global avg pool (awkward)

In [213]:
raw_gap = x.mean(dim=[2, 3])   # you must know dims 2,3 are spatial

# Einops version (readable)
ein_gap = reduce(x, 'b c h w -> b c', 'mean')

print(f"\nRaw and einops GAP match: {torch.allclose(raw_gap, ein_gap)}")


Raw and einops GAP match: True


## 2.3 `repeat` — Expand and Tile

In [214]:
# Basic repeat
from einops import repeat

v = torch.tensor([1., 2., 3.])   # (3,)

### Repeat to make a matrix: each row is the same vector

In [215]:
mat = repeat(v, 'd -> n d', n=4)    # (4, 3)
show(mat, "repeat v to (4,3)")

repeat v to (4,3): shape=(4, 3), dtype=torch.float32
tensor([[1., 2., 3.],
        [1., 2., 3.],
        [1., 2., 3.],
        [1., 2., 3.]])



### Add batch dim by repeating

In [216]:
template = torch.randn(10, 10)  # (H, W)
batch    = repeat(template, 'h w -> b h w', b=8)  # (8, H, W)
print(f"Add batch dim: {template.shape} → {batch.shape}")

Add batch dim: torch.Size([10, 10]) → torch.Size([8, 10, 10])


### CLS token in Transformers
- ViT prepends a learnable [CLS] token to the patch sequence

In [217]:
cls_token = torch.randn(1, 1, 512)       # (1, 1, embed)
B = 16
cls_batch = repeat(cls_token, '1 1 d -> b 1 d', b=B)  # (16, 1, 512)
print(f"\nCLS token broadcast: {cls_token.shape} → {cls_batch.shape}")


CLS token broadcast: torch.Size([1, 1, 512]) → torch.Size([16, 1, 512])


### Then concatenate with patches

In [218]:
patches = torch.randn(B, 196, 512)   # 14×14=196 patches
seq = torch.cat([cls_batch, patches], dim=1)  # (16, 197, 512)
print(f"Full sequence with CLS: {seq.shape}")

Full sequence with CLS: torch.Size([16, 197, 512])


### Tile: repeat n times along existing dim

In [219]:
x = torch.randn(3, 4)
tiled = repeat(x, 'h w -> h (n w)', n=3)  # tile width 3×
print(f"\nTile width 3×: {x.shape} → {tiled.shape}")


Tile width 3×: torch.Size([3, 4]) → torch.Size([3, 12])


### Positional encoding broadcast

In [220]:
pos_enc = torch.randn(1, 197, 512)   # (1, seq, embed)
pos_enc_batch = repeat(pos_enc, '1 n d -> b n d', b=B)
print(f"Pos encoding: {pos_enc.shape} → {pos_enc_batch.shape}")

Pos encoding: torch.Size([1, 197, 512]) → torch.Size([16, 197, 512])


## 2.4 Einops as a Debugging Tool

In [221]:
print("=== Shape Validation ===")

=== Shape Validation ===


### Wrong number of dimensions


In [222]:
try:
    x = torch.randn(3, 4)     # 2D
    rearrange(x, 'b h w -> b (h w)')  # expects 3D
except Exception as e:
    print(f"\n Wrong ndim: {e}")


 Wrong ndim:  Error while processing rearrange-reduction pattern "b h w -> b (h w)".
 Input tensor shape: torch.Size([3, 4]). Additional info: {}.
 Wrong shape: expected 3 dims. Received 2-dim tensor.


### Dimension mismatch (annotation helps locate the bug)

In [223]:
try:
    x = torch.randn(32, 128, 512)
    # Claim h=8 heads, but 512/8=64, so d=64 — this is fine
    rearrange(x, 'b n (h d) -> b h n d', h=8, d=128)  # 8*128 ≠ 512
except Exception as e:
    print(f"\n Dim mismatch: {e}")


 Dim mismatch:  Error while processing rearrange-reduction pattern "b n (h d) -> b h n d".
 Input tensor shape: torch.Size([32, 128, 512]). Additional info: {'h': 8, 'd': 128}.
 Shape mismatch, 512 != 1024


### Using einops to ASSERT shapes (acts as a runtime check!)


In [224]:
def check_shape(tensor, pattern, **axes):
    """Use rearrange as a shape assertion"""
    return rearrange(tensor, pattern, **axes)

x = torch.randn(32, 128, 512)  # expected: (batch=32, seq=128, embed=512)
try:
    x_checked = check_shape(x, 'b n d -> b n d')   # no-op but validates shape
    print(f"\n Shape assertion passed: {x_checked.shape}")
except Exception as e:
    print(f" Shape assertion failed: {e}")


 Shape assertion passed: torch.Size([32, 128, 512])


---
# PART 3: Einsum — Einstein Summation
---

Einsum answers: **what math happens across my tensor dimensions?**

**The one universal rule:**
- Indices **repeated on the left** but **absent on the right** → **summed over** (contracted)
- Indices **present on both sides** → **kept**
- Syntax: `'input1_indices, input2_indices -> output_indices'`

## 3.1 Einsum Fundamentals

In [225]:
print("The One Rule: repeated index → sum ")
print()

a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])

The One Rule: repeated index → sum 



### dot product: multiply element-wise then sum
- 'i,i->'  means: for each i, multiply a[i]*b[i], sum over i → scalar


In [226]:
dot = torch.einsum('i,i->', a, b)
print(f"Dot product 'i,i->': {dot.item()}  (= 1×4 + 2×5 + 3×6 = 32)")

Dot product 'i,i->': 32.0  (= 1×4 + 2×5 + 3×6 = 32)


### element-wise (keep i): 'i,i->i'

In [227]:
elemwise = torch.einsum('i,i->i', a, b)
print(f"Element-wise 'i,i->i': {elemwise}")

Element-wise 'i,i->i': tensor([ 4., 10., 18.])


### outer product: no shared index → all pairs
- 'i,j->ij' means: for each (i,j), compute a[i]*b[j]

In [228]:
outer = torch.einsum('i,j->ij', a, b)
print(f"\nOuter product 'i,j->ij':\n{outer}")


Outer product 'i,j->ij':
tensor([[ 4.,  5.,  6.],
        [ 8., 10., 12.],
        [12., 15., 18.]])


### sum a vector: 'i->'

In [229]:
total = torch.einsum('i->', a)
print(f"\nSum vector 'i->': {total.item()}")


Sum vector 'i->': 6.0


### scale: scalar times vector

In [230]:
s = torch.tensor(2.0)
scaled = torch.einsum(',i->i', s, a)
print(f"Scale ',i->i':                {scaled}")

Scale ',i->i':                tensor([2., 4., 6.])


## 3.2 Matrix Operations

In [231]:
A = torch.randn(3, 4)
B = torch.randn(4, 5)
print(f"A: {A.shape}, B: {B.shape}")

A: torch.Size([3, 4]), B: torch.Size([4, 5])


### Matrix multiplication
- 'ij,jk->ik': sum over shared j
- A[i,j] × B[j,k] → C[i,k]

In [232]:
C = torch.einsum('ij,jk->ik', A, B)
print(f"Matrix multiply 'ij,jk->ik': {A.shape} × {B.shape} → {C.shape}")
print(f"Matches torch.mm: {torch.allclose(C, torch.mm(A, B))}")


Matrix multiply 'ij,jk->ik': torch.Size([3, 4]) × torch.Size([4, 5]) → torch.Size([3, 5])
Matches torch.mm: True


### Matrix transpose

In [233]:
At = torch.einsum('ij->ji', A)
print(f"\nTranspose 'ij->ji': {A.shape} → {At.shape}")


Transpose 'ij->ji': torch.Size([3, 4]) → torch.Size([4, 3])


### Trace (sum of diagonal)

In [234]:
sq = torch.randn(4, 4)
trace = torch.einsum('ii->', sq)
print(f"\nTrace 'ii->': {trace.item():.4f}")
print(f"Matches torch.trace: {torch.allclose(trace, sq.trace())}")


Trace 'ii->': -2.2957
Matches torch.trace: True


### Element-wise product then sum (Frobenius inner product)

In [235]:
X = torch.randn(3, 4)
Y = torch.randn(3, 4)
frob = torch.einsum('ij,ij->', X, Y)
print(f"\nFrobenius 'ij,ij->': {frob.item():.4f}")
print(f"Matches (X*Y).sum(): {torch.allclose(frob, (X*Y).sum())}")


Frobenius 'ij,ij->': 2.3034
Matches (X*Y).sum(): True


### Matrix-vector product

In [236]:
M = torch.randn(3, 4)
v = torch.randn(4)
mv = torch.einsum('ij,j->i', M, v)   # (3,4) × (4,) → (3,)
print(f"\nMatrix-vector 'ij,j->i': {M.shape} × {v.shape} → {mv.shape}")


Matrix-vector 'ij,j->i': torch.Size([3, 4]) × torch.Size([4]) → torch.Size([3])


## 3.3 Batched Operations

### Batch matrix multiply

In [237]:
A = torch.randn(8, 3, 4)   # 8 matrices of shape (3×4)
B = torch.randn(8, 4, 5)   # 8 matrices of shape (4×5)
print(f"A: {A.shape}, B: {B.shape}")

A: torch.Size([8, 3, 4]), B: torch.Size([8, 4, 5])


### 'bij,bjk->bik': for each sample b, multiply A[b] @ B[b]

In [238]:
C = torch.einsum('bij,bjk->bik', A, B)  # (8, 3, 5)
print(f"Batched matmul 'bij,bjk->bik': {A.shape} × {B.shape} → {C.shape}")
print(f"Matches torch.bmm: {torch.allclose(C, torch.bmm(A, B))}")

Batched matmul 'bij,bjk->bik': torch.Size([8, 3, 4]) × torch.Size([8, 4, 5]) → torch.Size([8, 3, 5])
Matches torch.bmm: True


### Multi-batch dim (torch.bmm can't do this!)

In [239]:
A4 = torch.randn(4, 8, 3, 4)   # (batch1, batch2, M, K)
B4 = torch.randn(4, 8, 4, 5)   # (batch1, batch2, K, N)
C4 = torch.einsum('...ik,...kj->...ij', A4, B4)  # ... = any batch dims
print(f"\nMulti-batch '...ik,...kj->...ij': {A4.shape} × {B4.shape} → {C4.shape}")


Multi-batch '...ik,...kj->...ij': torch.Size([4, 8, 3, 4]) × torch.Size([4, 8, 4, 5]) → torch.Size([4, 8, 3, 5])


### Batched dot products

In [240]:
# Compare each pair of vectors in two batches
U = torch.randn(32, 128)  # (batch, dim)
V = torch.randn(32, 128)
# 'bd,bd->b': element-wise multiply then sum per sample
dots = torch.einsum('bd,bd->b', U, V)   # (32,): one dot per sample
print(f"\nBatched dots 'bd,bd->b': {U.shape} × {V.shape} → {dots.shape}")


Batched dots 'bd,bd->b': torch.Size([32, 128]) × torch.Size([32, 128]) → torch.Size([32])


### All-pairs dot products (similarity matrix)

In [241]:
# 'id,jd->ij': dot product between every pair (i,j)
sim_matrix = torch.einsum('id,jd->ij', U, V)  # (32, 32): i×j matrix
print(f"All-pairs dots 'id,jd->ij': {U.shape} × {V.shape} → {sim_matrix.shape}")

All-pairs dots 'id,jd->ij': torch.Size([32, 128]) × torch.Size([32, 128]) → torch.Size([32, 32])


## 3.4 Einsum in Transformers — Attention Mechanism

In [242]:
import math

B, H, N, D = 4, 8, 64, 64   # batch, heads, seq_len, head_dim

Q = torch.randn(B, H, N, D)   # queries
K = torch.randn(B, H, N, D)   # keys
V = torch.randn(B, H, N, D)   # values

### Step 1: Attention scores
- For each (batch, head), compute dot product between query i and key j
    - 'bhid,bhjd->bhij':
    - b=batch, h=head: kept
    - i=query position, j=key position: kept → 2D score matrix
    - d=head_dim: summed over (the contraction dimension)


In [243]:
scores = torch.einsum('bhid,bhjd->bhij', Q, K) / math.sqrt(D)
print(f"Attention scores:")
print(f" Q: {Q.shape}, K: {K.shape}")
print(f" einsum 'bhid,bhjd->bhij' → {scores.shape}")
print(f" = {B} batches × {H} heads × {N} queries × {N} keys\n")

Attention scores:
 Q: torch.Size([4, 8, 64, 64]), K: torch.Size([4, 8, 64, 64])
 einsum 'bhid,bhjd->bhij' → torch.Size([4, 8, 64, 64])
 = 4 batches × 8 heads × 64 queries × 64 keys



### Step 2: Softmax

In [244]:
weights = scores.softmax(dim=-1)  # over key dim j
print(f"Attention weights: {weights.shape}  (row-sums=1.0)")
print(f"  Row sum check: {weights[0,0,0].sum().item():.4f}\n")

Attention weights: torch.Size([4, 8, 64, 64])  (row-sums=1.0)
  Row sum check: 1.0000



### Step 3: Weighted sum of values
- 'bhij,bhjd->bhid':
  - b,h: batch dims, kept
  - i: query position, kept
  - j: key position, summed (contracted with weights)
  - d: value dim, kept

In [245]:
attended = torch.einsum('bhij,bhjd->bhid', weights, V)
print(f"Attended output:")
print(f" weights: {weights.shape}, V: {V.shape}")
print(f" einsum 'bhij,bhjd->bhid' → {attended.shape}")

Attended output:
 weights: torch.Size([4, 8, 64, 64]), V: torch.Size([4, 8, 64, 64])
 einsum 'bhij,bhjd->bhid' → torch.Size([4, 8, 64, 64])


### Full attention in ~5 lines

In [246]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores  = torch.einsum('bhid,bhjd->bhij', Q, K) / math.sqrt(d_k)
    weights = scores.softmax(dim=-1)
    output  = torch.einsum('bhij,bhjd->bhid', weights, V)
    return output, weights

out, w = scaled_dot_product_attention(Q, K, V)
print(f"\nFull attention: {out.shape}")


Full attention: torch.Size([4, 8, 64, 64])


## 3.5 Advanced Einsum Patterns

### Gram / Covariance matrix

In [247]:
# features: (N, d) — N data points in d-dimensional space
X = torch.randn(100, 32)
# Gram matrix: G[i,j] = X[i] · X[j] (all pairwise dot products)
G = torch.einsum('id,jd->ij', X, X)         # (100, 100)
# Covariance: C[a,b] = sum over samples of X[n,a]*X[n,b]
C = torch.einsum('na,nb->ab', X, X) / 100   # (32, 32) = X^T X / N
print(f"Gram matrix 'id,jd->ij': {X.shape} → {G.shape}")
print(f"Covariance 'na,nb->ab':  {X.shape} → {C.shape}")



Gram matrix 'id,jd->ij': torch.Size([100, 32]) → torch.Size([100, 100])
Covariance 'na,nb->ab':  torch.Size([100, 32]) → torch.Size([32, 32])


### Bilinear form: x^T A y

In [248]:
# Used in: metric learning, graph attention
n, m, k = 10, 5, 8
X = torch.randn(n, m)    # (N, M)
A = torch.randn(m, k)    # (M, K)
Y = torch.randn(n, k)    # (N, K)
# bilinear: for each sample i: X[i]^T @ A @ Y[i] → scalar per sample
bilinear = torch.einsum('im,mk,ik->i', X, A, Y)  # (N,)
print(f"\nBilinear 'im,mk,ik->i': ({n},{m})×({m},{k})×({n},{k}) → {bilinear.shape}")


Bilinear 'im,mk,ik->i': (10,5)×(5,8)×(10,8) → torch.Size([10])


### 3-way contraction (Graph Neural Network message passing)
- node_feat: (N, d), adj: (N, N), weight: (d, out)


In [249]:
N, d, out_d = 50, 16, 32
node_feat = torch.randn(N, d)
adj       = torch.rand(N, N)
W         = torch.randn(d, out_d)
# 'ij,jk,kl->il': adj × node_feat × weight in one shot
msg = torch.einsum('ij,jk,kl->il', adj, node_feat, W)  # (N, out_d)
print(f"\nGNN message pass 'ij,jk,kl->il': → {msg.shape}")


GNN message pass 'ij,jk,kl->il': → torch.Size([50, 32])


### Batched outer products

In [250]:
# Used in: FiLM conditioning, cross-attention
a = torch.randn(8, 16)   # (batch, dim_a)
b = torch.randn(8, 32)   # (batch, dim_b)
outer = torch.einsum('bi,bj->bij', a, b)  # (8, 16, 32)
print(f"Batched outer 'bi,bj->bij': {a.shape} × {b.shape} → {outer.shape}")

Batched outer 'bi,bj->bij': torch.Size([8, 16]) × torch.Size([8, 32]) → torch.Size([8, 16, 32])


### Diagonal extraction

In [251]:
M = torch.randn(5, 5)
diag_v = torch.einsum('ii->i', M)    # extract diagonal
print(f"\nDiagonal 'ii->i': {M.shape} → {diag_v.shape}")
print(f"Matches torch.diag: {torch.allclose(diag_v, torch.diag(M))}")


Diagonal 'ii->i': torch.Size([5, 5]) → torch.Size([5])
Matches torch.diag: True


# PART 4: Putting It All Together - Real ML Components

Here How Raw + Einops + Einsum complement each other in actual neural network code.

## 4.1 Full Multi-Head Attention Block

In [252]:
import torch
import torch.nn as nn
import math
from einops import rearrange

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.qkv     = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj= nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, N, D = x.shape

        # 1. Project to Q, K, V
        qkv = self.qkv(x)                           # (B, N, 3D)

        # 2. RAW: split last dim into 3 equal chunks
        q, k, v = qkv.chunk(3, dim=-1)              # each (B, N, D)

        # 3. EINOPS: split embed_dim into heads
        q = rearrange(q, 'b n (h d) -> b h n d', h=self.num_heads)
        k = rearrange(k, 'b n (h d) -> b h n d', h=self.num_heads)
        v = rearrange(v, 'b n (h d) -> b h n d', h=self.num_heads)

        # 4. EINSUM: compute attention scores (pure math)
        scores = torch.einsum('bhid,bhjd->bhij', q, k) * self.scale

        # 5. RAW: optional causal mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # 6. RAW: softmax + dropout
        attn = self.dropout(scores.softmax(dim=-1))

        # 7. EINSUM: weighted sum of values
        out = torch.einsum('bhij,bhjd->bhid', attn, v)

        # 8. EINOPS: merge heads back
        out = rearrange(out, 'b h n d -> b n (h d)')

        # 9. RAW: final projection
        return self.out_proj(out)


In [253]:
# Test it
B, N, D = 4, 32, 256
x   = torch.randn(B, N, D)
mha = MultiHeadAttention(embed_dim=D, num_heads=8)
out = mha(x)
print(f"MHA Input:  {x.shape}")
print(f"MHA Output: {out.shape}")
print(f"\nTool usage in MHA:")
print(f"  Raw   → chunk, masked_fill, softmax, dropout, Linear")
print(f"  Einops→ rearrange (split heads, merge heads)")
print(f"  Einsum→ attention scores, weighted value aggregation")

MHA Input:  torch.Size([4, 32, 256])
MHA Output: torch.Size([4, 32, 256])

Tool usage in MHA:
  Raw   → chunk, masked_fill, softmax, dropout, Linear
  Einops→ rearrange (split heads, merge heads)
  Einsum→ attention scores, weighted value aggregation


## 4.2 Vision Transformer (ViT) Patch Embedding

In [254]:
from einops import rearrange, repeat

class PatchEmbedding(nn.Module):
    """Splits images into patches and linearly embeds them."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        n_patches = (img_size // patch_size) ** 2
        patch_dim  = in_channels * patch_size * patch_size

        self.projection = nn.Linear(patch_dim, embed_dim)
        self.cls_token  = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed  = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim))

    def forward(self, x):
        B = x.shape[0]
        p = self.patch_size

        # EINOPS: patchify  (b c (h p1) (w p2) → b (h w) (c p1 p2))
        patches = rearrange(x, 'b c (h p1) (w p2) -> b (h w) (c p1 p2)',
                            p1=p, p2=p)

        # RAW: linear projection
        tokens = self.projection(patches)   # (B, n_patches, embed_dim)

        # EINOPS: expand CLS token to batch size
        cls = repeat(self.cls_token, '1 1 d -> b 1 d', b=B)

        # RAW: prepend CLS token
        tokens = torch.cat([cls, tokens], dim=1)   # (B, n_patches+1, embed_dim)

        # RAW: add positional encoding (broadcast)
        tokens = tokens + self.pos_embed

        return tokens


In [255]:
# Test
imgs   = torch.randn(4, 3, 224, 224)
pe     = PatchEmbedding(img_size=224, patch_size=16, embed_dim=768)
tokens = pe(imgs)
print(f"Input images: {imgs.shape}")
print(f"Output tokens: {tokens.shape}")
print(f"  = {4} images, {196+1} tokens (196 patches + 1 CLS), {768}-dim each")

Input images: torch.Size([4, 3, 224, 224])
Output tokens: torch.Size([4, 197, 768])
  = 4 images, 197 tokens (196 patches + 1 CLS), 768-dim each


## 4.3 CNN: Feature Map Operations

In [256]:
from einops import rearrange, reduce

# Simulated CNN feature maps after conv layers
feat = torch.randn(16, 256, 14, 14)   # (batch, channels, H, W)

print("CNN Feature Map Operations")

# Global Average Pooling (GAP)
gap = reduce(feat, 'b c h w -> b c', 'mean')
print(f"GAP (for classifier): {feat.shape} → {gap.shape}")

# Spatial Pyramid Pooling
pool_4x4 = reduce(feat, 'b c (h ph) (w pw) -> b c h w', 'max', ph=2, pw=2)  # 14→7
pool_2x2 = reduce(feat, 'b c (h ph) (w pw) -> b c h w', 'max', ph=7, pw=7)  # 14→2
pool_1x1 = reduce(feat, 'b c h w -> b c 1 1', 'max')                          # 14→1
print(f"\nSpatial Pyramid:")
print(f"  4×4 level:  {feat.shape} → {pool_4x4.shape}")
print(f"  2×2 level:  {feat.shape} → {pool_2x2.shape}")
print(f"  1×1 level:  {feat.shape} → {pool_1x1.shape}")

# Channel shuffle (ShuffleNet operation)
# Split channels into groups and interleave them
g = 4   # groups
shuffled = rearrange(feat, 'b (g c) h w -> b (c g) h w', g=g)
print(f"\nChannel shuffle (g={g}): {feat.shape} → {shuffled.shape}")

# Convert to sequence for attention
# Used in: Vision Transformer, DETR
seq = rearrange(feat, 'b c h w -> b (h w) c')   # spatial positions as tokens
print(f"\nSpatial to sequence: {feat.shape} → {seq.shape}")
print(f"  = {16} images × {14*14} spatial positions × {256} channels")

# Gram matrix for style transfer
# G[i,j] = correlation between feature maps i and j
flat  = feat.reshape(16, 256, -1)   # (B, C, H*W)
gram  = torch.einsum('bci,bdi->bcd', flat, flat) / (14*14)  # (B, C, C)
print(f"\nGram matrix 'bci,bdi->bcd': {flat.shape} → {gram.shape}")

CNN Feature Map Operations
GAP (for classifier): torch.Size([16, 256, 14, 14]) → torch.Size([16, 256])

Spatial Pyramid:
  4×4 level:  torch.Size([16, 256, 14, 14]) → torch.Size([16, 256, 7, 7])
  2×2 level:  torch.Size([16, 256, 14, 14]) → torch.Size([16, 256, 2, 2])
  1×1 level:  torch.Size([16, 256, 14, 14]) → torch.Size([16, 256, 1, 1])

Channel shuffle (g=4): torch.Size([16, 256, 14, 14]) → torch.Size([16, 256, 14, 14])

Spatial to sequence: torch.Size([16, 256, 14, 14]) → torch.Size([16, 196, 256])
  = 16 images × 196 spatial positions × 256 channels

Gram matrix 'bci,bdi->bcd': torch.Size([16, 256, 196]) → torch.Size([16, 256, 256])


---
# PART 5: Side-by-Side Comparisons
---

In [257]:
feat = torch.randn(8, 3, 32, 32)  # (batch, channels, H, W)

### Operation 1: BCHW → BHWC

In [258]:
raw_1  = feat.permute(0, 2, 3, 1)                        # what does 0,2,3,1 mean??
ein_1  = rearrange(feat, 'b c h w -> b h w c')           # self-explanatory
print(f"   Raw:    feat.permute(0,2,3,1) → {raw_1.shape}")
print(f"   Einops: rearrange(..., 'b c h w -> b h w c') → {ein_1.shape}")
print(f"   Same:   {torch.allclose(raw_1, ein_1)}")

   Raw:    feat.permute(0,2,3,1) → torch.Size([8, 32, 32, 3])
   Einops: rearrange(..., 'b c h w -> b h w c') → torch.Size([8, 32, 32, 3])
   Same:   True


### Operation 2: Global Average Pool

In [259]:
## spatial → vector
raw_2  = feat.mean(dim=[2, 3])                           # need to know dims 2,3 are spatial
ein_2  = reduce(feat, 'b c h w -> b c', 'mean')          # explicit
print(f"   Raw:    feat.mean(dim=[2,3]) → {raw_2.shape}")
print(f"   Einops: reduce(..., 'b c h w -> b c', 'mean') → {ein_2.shape}")
print(f"   Same:   {torch.allclose(raw_2, ein_2)}")


   Raw:    feat.mean(dim=[2,3]) → torch.Size([8, 3])
   Einops: reduce(..., 'b c h w -> b c', 'mean') → torch.Size([8, 3])
   Same:   True



### Opeartion 3: Batch Matrix Multiply

In [260]:
A = torch.randn(4, 8, 16); B = torch.randn(4, 16, 32)
raw_3  = torch.bmm(A, B)                                 # only works for 3D
ein_3  = torch.einsum('bij,bjk->bik', A, B)              # works for any #dims
print(f"   Raw:    torch.bmm(A,B) → {raw_3.shape}")
print(f"   Einsum: einsum('bij,bjk->bik') → {ein_3.shape}")
print(f"   Same:   {torch.allclose(raw_3, ein_3)}")

   Raw:    torch.bmm(A,B) → torch.Size([4, 8, 32])
   Einsum: einsum('bij,bjk->bik') → torch.Size([4, 8, 32])
   Same:   True


### Operation 4: Dot Product Per Sample

In [261]:
U = torch.randn(32, 128); V = torch.randn(32, 128)
raw_4  = (U * V).sum(dim=1)                              # two operations
ein_4  = torch.einsum('bd,bd->b', U, V)                  # one expression
print(f"   Raw:    (U*V).sum(dim=1) → {raw_4.shape}")
print(f"   Einsum: einsum('bd,bd->b') → {ein_4.shape}")
print(f"   Same:   {torch.allclose(raw_4, ein_4)}")

   Raw:    (U*V).sum(dim=1) → torch.Size([32])
   Einsum: einsum('bd,bd->b') → torch.Size([32])
   Same:   True


---
# Summary
---

| Tool | Question answered | Key functions | Use for |
|---|---|---|---|
| **Raw** | How do I move data? | `.view`, `.permute`, `.cat`, `.squeeze` | Simple, obvious ops |
| **Einops** | How is my tensor structured? | `rearrange`, `reduce`, `repeat` | Complex reshaping, pooling, expanding |
| **Einsum** | What math happens across dims? | `torch.einsum('ij,jk->ik', A, B)` | Matrix mult, attention, contractions |

**They are not competing — they have non-overlapping jobs.**

The best deep learning code uses all three together:
- **Einops** moves data around
- **Einsum** computes on data  
- **Raw** handles the simple, standard cases